In [ ]:
from dotenv import load_dotenv
from jsonschema.benchmarks.contains import end
from langchain.chat_models import init_chat_model
from langchain_community.chat_models import ChatZhipuAI
from openai.types.beta import assistant

load_dotenv(override=True)

# ZHIPU_API_KEY = os.getenv("ZHIPUAI_API_KEY")
# ZHIPU_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")
model = ChatZhipuAI(
    model="glm-4.5-Air"
)
message = [
    {"role": "system", "content": "你是一个ai助手"},
    {"role": "user", "content": "2+2=?"}
]
res = model.invoke(message)
print(res.content)

# 对象格式
## JSON格式

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

messages = [
    SystemMessage("你是一个善于给出通俗易懂解释的AI助手"),
    HumanMessage("你好"),
    AIMessage("你好！我能帮你什么？"),
    HumanMessage("什么是机器学习"),
]
response = model.invoke(messages)
print(response.content)

# HumanMessage 消息列表
## 练习

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.chat_models import ChatZhipuAI
load_dotenv(override=True)

model = ChatZhipuAI(
    model="glm-4.5-Air"
)

messages = [
        SystemMessage(content="你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用'第二个人'这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\": [{\"name\": \"...\", \"claim\": \"...\"}]}"),
    HumanMessage(content="我认为 1+1=2", name="Bob"),
    HumanMessage(content="我认为 1+1>2", name="Tom"),
    HumanMessage(content="请列出谁说了什么，不要判断对错。", name="audience")
]

res = model.invoke(messages)
print(res.content)


# AIMessage 参数列表

In [ ]:
# AIMessage("你好~")
# AIMessage(content="北京今天晴天，温度 15°C")
from rich import print as rprint
messages = [
    SystemMessage("你叫小智，是一名助人为乐的助手。"),
    HumanMessage("你好，好久不见，请介绍下你自己。")
]
res = model.invoke(messages)
# rprint(res)
rprint(res.id)

# 历史保存
## 第一次 res 存入model
### 然后将res的内容保存到conversation中,让数组中存入，并且assisation关键字记住你
### #缺点就是费token然后如果有N条数据进来就会崩溃

In [ ]:
conversation = []

#第一次
conversation.append({"role" : "user","content" : "我叫一个小混混"})
res = model.invoke(conversation)

#
conversation.append({"role" : "assistant","content" : res.content})

#
conversation.append({"role" : "user","content" :"我叫什么名字, 你认为我该怎么改进呢"})
res2 = model.invoke(conversation)

rprint(res2.content)

## 解决方案：仅保留最近N轮对话，丢弃更早的对话

In [49]:

def keep_recent(message,max_pairs=3):
    system_msgs = [m for m in message if m.get("role") == "system"]
    conversation_msgs = [m for m in message if m.get("role") != "assistant"]

    recent_msgs = conversation_msgs[-(max_pairs *2):]
    return system_msgs + recent_msgs

# 初始化
long_conversation = [{"role": "system", "content": "你是 厨师"}]

long_conversation.append({"role": "user","content":"什么是回锅肉，一句话表示"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role" : "assistant" , "content":r1.content})

#第二轮
long_conversation.append({"role":"user","content" : "什么是狮子头？一句话表示"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role" : "assistant" , "content":r2.content})
#第三轮
long_conversation.append({"role":"user","content" : "什么是蚂蚁上树？一句话表示"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role" : "assistant" , "content":r3.content})

print(f"原始消息数: {len(long_conversation)}")

optimized = keep_recent(long_conversation, max_pairs=1)

print(f"优化后消息数: {len(optimized)}")
optimized.append({"role": "user", "content": "我第一个问题的答案是什么？"})
optimized.append({"role": "user", "content": "我问了几个问题,分别是什么问题？"})
res4 = model.invoke(optimized)
rprint(res4.content)


原始消息数: 7
优化后消息数: 3


您问了4个问题：
1. 什么是狮子头？一句话表示
2. 什么是蚂蚁上树？一句话表示
3. 我第一个问题的答案是什么？
4. 我问了几个问题,分别是什么问题？

# 多轮对话机器人

In [53]:
from langchain.chat_models import init_chat_model
import os
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)

MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"

model = ChatZhipuAI(
    model="glm-4.5-Air"
)
#初始化大模型的角色
messages = [{
    "role": "system","content":"你是一位很关心人的小姐姐呦，具有台湾腔的，你最喜欢发颜文字了在最后"
}]

rprint(f"❤ 请输入问题, 输入{EXIT_WORD} 结束对话 ❤ \n")

i=1
while True:
    rprint("\n", "=" * 10,f'-> 第一{i}轮开始 <-',"="*10,"\n")
    user_input = input("🙋‍ 请输入你的难题: ")

    #exit 判断
    if user_input == EXIT_WORD:
        print("已退出😊")
        break
    #把输入的问题进行记录
    messages.append({"role": "user", "content": user_input})

    rprint("小姐姐 :", end="",flush=True)

    reply_content=""

    memory_messages = keep_recent(messages,MAX_PAIRS_HISTORY)
    # 控制发送给模型的消息长度
    for chunk in model.stream(memory_messages):
        if chunk.content:
            rprint(chunk.content, end="", flush=True)
            reply_content += chunk.content
    rprint("\n", "=" * 10, f'-> 第 {i} 轮对话结束 <-', "=" * 10, "\n")
    i += 1

    # 追加 AI 回复
    messages.append({"role":"assistant","content":reply_content})

❤ 请输入问题, 输入quit 结束对话 ❤

========== -> 第一1轮开始 <- ==========

小姐姐 :

喔~当然知道啦！粽子是很有名的中国传统美食耶！特别是在端午节的时候，大家都会包粽子来吃呢～

粽子是用糯米包裹各种馅料，然后用竹叶或芦苇叶包起来的三角形状食物，有甜的也有咸的口味。甜粽通常会加蜜枣、豆沙或八宝；咸粽则有猪肉、咸蛋黄或香菇等不同馅料，每种都好好吃喔！

你有特别喜欢吃哪种口味的粽子吗？还是有自己包过粽子呢？记得要多喝水、慢慢吃粽子，不要噎到唷！( ´ ω ` )ﾉ
 ========== -> 第 1 轮对话结束 <- ========== 



========== -> 第一2轮开始 <- ==========

已退出😊


In [ ]:
quit
